## Objective

This notebook prepares final BI-ready datasets that translate analytical outputs into formats suitable for visualization, reporting, and stakeholder usage. Earlier notebooks focused on modeling and segmentation; this step formalizes the data handoff to business intelligence tools.

### Star Schema Design

The output follows a star schema pattern:

- **Fact table:** Transaction-level records for time-based analysis.
- **Dimension table:** Customer-level attributes for slicing and filtering.
- **Data dictionary:** Field definitions for consistent interpretation.

### Outputs

1. Transaction fact table (Parquet + CSV).
2. Customer dimension table with behavioral features, segmentation, and CLV estimates (Parquet + CSV).
3. Data dictionary documenting all fields (Parquet + CSV).
4. Date dimension table for Power BI time intelligence (Parquet + CSV).

## Load Cleaned Data and Build Star Schema for Power BI

In [ ]:
import pandas as pd
import numpy as np

# Load all input datasets
txns_path = "/kaggle/input/online-retail-clean/transactions_clean.csv"
txns_data = pd.read_csv(txns_path, parse_dates=["InvoiceDate"])

customers_features_path = "/kaggle/input/customer-features/customers_features.csv"
customers_features = pd.read_csv(customers_features_path, parse_dates=["FirstPurchase", "LastPurchase"])

customer_segments_path = "/kaggle/input/datasets/customer-segments/customer_segments.csv"
customer_segments = pd.read_csv(customer_segments_path)

print(f"Transactions: {txns_data.shape}")
print(f"Customer features: {customers_features.shape}")
print(f"Customer segments: {customer_segments.shape}")

# Verify column names for consistency
print(f"\nCustomer features columns: {customers_features.columns.tolist()}")
print(f"Customer segments columns: {customer_segments.columns.tolist()}")

Transactions: (804008, 9)
Customer features: (5840, 11)
Customer segments: (5840, 12)

Customer features columns: ['CustomerID', 'Frequency', 'Monetary', 'AvgOrderValue', 'AvgItemsPerOrder', 'FirstPurchase', 'LastPurchase', 'RecencyDays', 'TenureDays', 'CustomerAgeDays', 'PurchaseRateMonthly']
Customer segments columns: ['CustomerID', 'RFM_Segment', 'RFM_Score', 'Cluster', 'Cluster_Label', 'RecencyDays', 'Frequency', 'Monetary', 'AvgOrderValue', 'PurchaseRateMonthly', 'CLV_12m', 'CLV_6m_pred']


## Build Star Schema

The customer dimension table is built by merging customer features with segmentation results. The fact table preserves transaction-level detail with a date key for time intelligence.

In [2]:
# Identify overlapping columns between features and segments
# Customer segments may contain redundant copies of features already in the features table
segments_unique_columns = customer_segments.columns.difference(customers_features.columns).tolist()
segments_unique_columns.append("CustomerID")

print(f"Unique columns from segments: {segments_unique_columns}")

# Build customer dimension by merging features with segment assignments
dimension_customers = customers_features.merge(
    customer_segments[segments_unique_columns],
    on = "CustomerID",
    how = "left"
)

# Build fact table with date key
fact_table = txns_data.copy()
fact_table["Date"] = fact_table["InvoiceDate"].dt.date
fact_table["DateKey"] = fact_table["InvoiceDate"].dt.strftime("%Y%m%d").astype(int)

# Create date dimension for Power BI time intelligence
date_range = pd.date_range(
    start = fact_table["Date"].min(),
    end = fact_table["Date"].max(),
    freq = "D"
)

date_dimension = pd.DataFrame({
    "DateKey": date_range.strftime("%Y%m%d").astype(int),
    "Date": date_range.date,
    "Year": date_range.year,
    "Quarter": date_range.quarter,
    "Month": date_range.month,
    "MonthName": date_range.strftime("%B"),
    "DayOfWeek": date_range.dayofweek,
    "DayOfWeekName": date_range.strftime("%A"),
    "IsWeekend": (date_range.dayofweek >= 5).astype(int)
})

print(f"\nDimension customers: {dimension_customers.shape}")
print(f"Fact table: {fact_table.shape}")
print(f"Date dimension: {date_dimension.shape}")

Unique columns from segments: ['CLV_12m', 'CLV_6m_pred', 'Cluster', 'Cluster_Label', 'RFM_Score', 'RFM_Segment', 'CustomerID']

Dimension customers: (5840, 17)
Fact table: (804008, 11)
Date dimension: (739, 9)


## Validate Star Schema Integrity

Before exporting, the star schema relationships are validated to ensure:

- All fact table CustomerIDs exist in the dimension table.
- No duplicate CustomerIDs in dimension table.
- Date keys align between fact and date dimension.

In [3]:
# Validation checks
fact_customers = set(fact_table["CustomerID"].unique())
dimension_customers_ids = set(dimension_customers["CustomerID"].unique())

# Check 1: All fact customers exist in dimension
missing_in_dimension = fact_customers - dimension_customers_ids
print(f"Customers in fact table missing from dimension: {len(missing_in_dimension):,}")

# Check 2: No duplicate CustomerIDs in dimension
duplicate_ids = dimension_customers["CustomerID"].duplicated().sum()
print(f"Duplicate CustomerIDs in dimension: {duplicate_ids:,}")

# Check 3: Date keys align
fact_date_keys = set(fact_table["DateKey"].unique())
dim_date_keys = set(date_dimension["DateKey"].unique())
missing_dates = fact_date_keys - dim_date_keys
print(f"Fact dates missing from date dimension: {len(missing_dates):,}")

# Check 4: No negative quantities in fact table
negative_quantities = (fact_table["Quantity"] < 0).sum()
print(f"Negative quantity rows in fact table: {negative_quantities:,}")

# Summary
print(f"\n=== Star Schema Validation ===")
print(f"Fact table customers: {len(fact_customers):,}")
print(f"Dimension table customers: {len(dimension_customers_ids):,}")
print(f"Date dimension days: {len(date_dimension):,}")
print(f"All validations passed: {len(missing_in_dimension) == 0 and duplicate_ids == 0 and len(missing_dates) == 0}")

Customers in fact table missing from dimension: 0
Duplicate CustomerIDs in dimension: 0
Fact dates missing from date dimension: 0
Negative quantity rows in fact table: 0

=== Star Schema Validation ===
Fact table customers: 5,840
Dimension table customers: 5,840
Date dimension days: 739
All validations passed: True


## Export BI-Ready Tables

All tables are exported in both Parquet (for Power BI efficiency) and CSV (for maximum compatibility).

In [4]:
# Export fact table
fact_table.to_parquet("fact_transactions.parquet", index = False)
fact_table.to_csv("fact_transactions.csv", index = False)
print("Saved: fact_transactions.parquet and .csv")

# Export customer dimension table
dimension_customers.to_parquet("dimension_customers.parquet", index = False)
dimension_customers.to_csv("dimension_customers.csv", index = False)
print("Saved: dimension_customers.parquet and .csv")

# Export date dimension
date_dimension.to_parquet("dimension_date.parquet", index = False)
date_dimension.to_csv("dimension_date.csv", index = False)
print("Saved: dimension_date.parquet and .csv")

Saved: fact_transactions.parquet and .csv
Saved: dimension_customers.parquet and .csv
Saved: dimension_date.parquet and .csv


## Create Data Dictionary

The data dictionary documents every field in the exported tables with clear definitions and intended usage. This ensures consistent interpretation across technical and non-technical audiences.

In [5]:
# Complete data dictionary with accurate column names and definitions
data_dict = [
    # Fact table
    ("fact_transactions", "Invoice", "Order/invoice identifier (string)"),
    ("fact_transactions", "StockCode", "Product stock code (string)"),
    ("fact_transactions", "Description", "Product description (string)"),
    ("fact_transactions", "Quantity", "Quantity purchased (positive integers only after cleaning)"),
    ("fact_transactions", "InvoiceDate", "Date/time of transaction (datetime)"),
    ("fact_transactions", "Price", "Unit price in £ (float)"),
    ("fact_transactions", "CustomerID", "Customer identifier (float, join key to dimension_customers)"),
    ("fact_transactions", "Country", "Country of transaction (string)"),
    ("fact_transactions", "TotalPrice", "Transaction revenue = Quantity × Price (float)"),
    ("fact_transactions", "Date", "Calendar date of transaction (date)"),
    ("fact_transactions", "DateKey", "Integer date key for date dimension join (YYYYMMDD)"),
    
    # Customer dimension
    ("dimension_customers", "CustomerID", "Customer identifier (join key to fact_transactions)"),
    ("dimension_customers", "Frequency", "Number of unique orders in customer history"),
    ("dimension_customers", "Monetary", "Total historical revenue in £"),
    ("dimension_customers", "AvgOrderValue", "Average revenue per order in £"),
    ("dimension_customers", "AvgItemsPerOrder", "Average number of items per order"),
    ("dimension_customers", "FirstPurchase", "Date of first observed purchase"),
    ("dimension_customers", "LastPurchase", "Date of most recent observed purchase"),
    ("dimension_customers", "RecencyDays", "Days since last purchase at snapshot date"),
    ("dimension_customers", "TenureDays", "Days between first and last purchase"),
    ("dimension_customers", "CustomerAgeDays", "Days since first purchase at snapshot date"),
    ("dimension_customers", "PurchaseRateMonthly", "Orders per month (Frequency / (TenureDays/30 + 1/30)); zero-tenure customers set to Frequency"),
    ("dimension_customers", "CLV_12m", "12-month probabilistic CLV from BG/NBD + Gamma-Gamma (null for one-time buyers)"),
    ("dimension_customers", "CLV_6m_pred", "ML-predicted 6-month revenue from Random Forest"),
    ("dimension_customers", "Cluster", "K-Means cluster assignment (0-3)"),
    ("dimension_customers", "Cluster_Label", "Business label for K-Means cluster"),
    ("dimension_customers", "RFM_Score", "Composite RFM score (R×100 + F×10 + M)"),
    ("dimension_customers", "RFM_Segment", "Business-friendly RFM segment label"),
    
    # Date dimension
    ("dimension_date", "DateKey", "Integer date key (YYYYMMDD)"),
    ("dimension_date", "Date", "Calendar date"),
    ("dimension_date", "Year", "Calendar year"),
    ("dimension_date", "Quarter", "Calendar quarter (1-4)"),
    ("dimension_date", "Month", "Calendar month (1-12)"),
    ("dimension_date", "MonthName", "Full month name"),
    ("dimension_date", "DayOfWeek", "Day of week (0 = Monday, 6 = Sunday)"),
    ("dimension_date", "DayOfWeekName", "Full day of week name"),
    ("dimension_date", "IsWeekend", "Boolean flag: 1 if weekend, 0 if weekday"),
]

data_dictionary = pd.DataFrame(data_dict, columns = ["Table", "Column", "Definition"])

# Export data dictionary
data_dictionary.to_parquet("data_dictionary.parquet", index = False)
data_dictionary.to_csv("data_dictionary.csv", index = False)
print("Saved: data_dictionary.parquet and .csv")

# Display complete dictionary
print(f"\nData dictionary records: {len(data_dictionary)}")
print(f"Fact table fields: {data_dictionary[data_dictionary['Table'] == 'fact_transactions'].shape[0]}")
print(f"Dimension table fields: {data_dictionary[data_dictionary['Table'] == 'dimension_customers'].shape[0]}")
print(f"Date dimension fields: {data_dictionary[data_dictionary['Table'] == 'dimension_date'].shape[0]}")

data_dictionary

Saved: data_dictionary.parquet and .csv

Data dictionary records: 37
Fact table fields: 11
Dimension table fields: 17
Date dimension fields: 9


,Table,Column,Definition
0,fact_transactions,Invoice,Order/invoice identifier (string)
1,fact_transactions,StockCode,Product stock code (string)
2,fact_transactions,Description,Product description (string)
3,fact_transactions,Quantity,Quantity purchased (positive integers only aft...
4,fact_transactions,InvoiceDate,Date/time of transaction (datetime)
5,fact_transactions,Price,Unit price in £ (float)
6,fact_transactions,CustomerID,"Customer identifier (float, join key to dimens..."
7,fact_transactions,Country,Country of transaction (string)
8,fact_transactions,TotalPrice,Transaction revenue = Quantity × Price (float)
9,fact_transactions,Date,Calendar date of transaction (date)


## Power BI Integration Notes

**Star Schema Relationships:**
- `fact_transactions.CustomerID` → `dimension_customers.CustomerID` (many-to-one).
- `fact_transactions.DateKey` → `dimension_date.DateKey` (many-to-one).

**Recommended Power BI setup:**
1. Import all tables.
2. Set relationships as specified above.
3. Mark `dimension_date` as a date table.
4. Use `dimension_customers.RFM_Segment` and `Cluster_Label` for slicers.
5. Use `fact_transactions.TotalPrice` for revenue measures.
6. Use `dimension_customers.CLV_12m` and `CLV_6m_pred` for value-based filtering.

**Coverage notes:**
- `CLV_12m` is null for 1,669 one-time buyers (excluded from Gamma-Gamma model).
- `CLV_6m_pred` is null for 908 customers acquired after the ML cutoff date.
- These nulls are intentional and should not be treated as data quality issues.